In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/produkcja_clean.csv')

display(df.shape)
display(df.dtypes)
display(df.isna().sum())
display(df.duplicated().sum())
display(df['Machine failure'].value_counts())

In [ ]:
X = df.drop(columns = ['Machine failure'])
y = df['Machine failure']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Rozkład klas:")
print(y.value_counts())
print(y.value_counts(normalize=True).round(3) * 100)



In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
y_train = y_train_raw.copy()
y_test = y_test_raw.copy()

assert X_train_raw.index.equals(y_train_raw.index), "Indeksy X_train i y_train nie są zgodne"
assert X_test_raw.index.equals(y_test_raw.index), "Indeksy X_test i y_test nie są zgodne"
assert len(X_train_raw) == len(y_train_raw), "X_train_raw i y_train_raw mają różną liczbę rekordów."
assert len(X_test_raw) == len(y_test_raw), "X_test_raw i y_test_raw mają różną liczbę rekordów."
assert np.isclose(y_train_raw.mean(), y_test_raw.mean(), atol = 0.001), "Proporcje klas w train i test różnią się bardziej niż oczekiwano."
print(f"Train : {X_train.shape[0]} rekordów")
print(f"Test : {X_test.shape[0]} rekordów")
print(f"Awarie w train: {y_train.sum()} ({y_train.mean() * 100:.1f}%)")
print(f"Awarie w test: {y_test.sum()} ({y_test.mean() * 100:.1f}%)")
print("Kontrola podziału danych zakończona poprawnie.")
print(f"Liczba rekordów treningowych: {len(X_train_raw)}")
print(f"Liczba rekordów testowych: {len(X_test_raw)}")

In [ ]:
cols_to_scale = [
    'Process temperature [K]',
    'Temperature difference',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Power [W]',
    'Tool wear [min]'
]

scaler = StandardScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train_raw[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test_raw[cols_to_scale])

assert X_train.index.equals(X_train_raw.index), "Skalowanie zmieniło indeksy zbioru treningowego."

assert X_test.index.equals(X_test_raw.index), "Skalowanie zmieniło indeksy zbioru testowego."

assert X_train.shape == X_train_raw.shape, ("Skalowany i surowy zbiór treningowy mają różne wymiary."
)

assert X_test.shape == X_test_raw.shape, ("Skalowany i surowy zbiór testowy mają różne wymiary."
)

assert (y_test.reset_index(drop=True) == y_test_raw.reset_index(drop=True)).all(), "Wartości targetu w obu zbiorach testowych są różne."

assert (y_train.reset_index(drop=True) == y_train_raw.reset_index(drop=True)).all(), "Wartości targetu w obu zbiorach treningowych są różne."

print("Kontrola skalowania danych zakończona poprawnie.")

In [ ]:
scaled_df = pd.DataFrame(
    X_train[cols_to_scale],
    columns = cols_to_scale
)
display(scaled_df.describe().round(3))

In [ ]:
print(scaler.mean_)
print(scaler.scale_)

In [ ]:
X_train.to_csv("../data/processed/X_train_scaled.csv", index=False)
X_test.to_csv("../data/processed/X_test_scaled.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)